# Load Models

In the previous notebook we saved the large model. In this notebook we have a look on how to reload it.

## Setup & Data

In [4]:
# Import packages
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import warnings

warnings.filterwarnings("ignore")

tf.keras.backend.set_floatx("float64")

# Define random seed for whole notebook
RSEED = 42

In [5]:
# Load data
df = pd.read_csv("../data/boston.csv")

# Define target
y = df.pop("MEDV")

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RSEED)

In [6]:
# Scale numerical features
# Scale numerical values
col_scale = [
    "CRIM",
    "ZN",
    "INDUS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "TAX",
    "PTRATIO",
    "LSTAT",
]

scaler = MinMaxScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

# Convert to np array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

The SavedModel format is a directory containing a protobuf binary and a TensorFlow checkpoint. Inspect the saved model directory:

In [1]:
!ls saved_model

my_large_model.keras


In [2]:
!ls saved_model/my_large_model.keras

saved_model/my_large_model.keras


## SavedModel format

Reload a fresh Keras model from the saved model:

In [7]:
# Load the saved model
with tf.device("/cpu:0"):
    new_large_model = tf.keras.models.load_model("saved_model/my_large_model.keras")

# Check its architecture
new_large_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 512)            │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,385,412 (18.20 MB)

 Trainable params: 795,137 (6.07 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,590,275 (12.13 MB)

The restored model is compiled with the same arguments as the original model. Try running evaluate and predict with the loaded model:

In [8]:
# Evaluate the restored model
with tf.device("/cpu:0"):
    loss, mse = new_large_model.evaluate(X_test, y_test, verbose=2)
print(f"Model MSE: {mse}")

4/4 - 0s - 32ms/step - loss: 21.4038 - mse: 528.1998
Model MSE: 528.1998077053712


The Model MSE is higher than in the notebook before, even though we are using the same data-split. We are using the same model and the same data. Therefore, we would assume that we also receive the same MSE. 

> **Exercise:** Can you find out where this notebook varies from the procedure in the notebook before? It might help to have a look at the values in X_train in both notebooks.

<details><summary>
Click here for a hint...
</summary>
Check out how the data were preprocessed. Did both notebooks use the same scaler?
</details>

The difference is that in notebook 3 we're using `StandardScaler` and in this notebook we're using `MinMaxScaler`. A trained neural network should always use the same scaler used to train it, otherwise it breaks the arithmetic.